# **DATA EXPLORATION AND PREPROCESSING**

---

## **1. Import Libraries**

In [1]:
import pandas as pd
import numpy as np
import sys
import os

## **2. Load Raw Data**

Trong giai đoạn đầu của quá trình khám phá dữ liệu (EDA), chúng em tiến hành tải dữ liệu thô (raw data) từ tệp CSV được lưu trữ trong cấu trúc thư mục dự án (`data/raw`).

In [2]:
occupancy_path = "../../data/raw/Room_Occupancy.csv"
df_occupancy = pd.read_csv(occupancy_path)

Dữ liệu được đọc từ tệp `Room_Occupancy.csv` và được khởi tạo dưới dạng cấu trúc bảng lưu vào biến DataFrame `df_energy`.

## **3. Basic Dataset Overview**

### **3.1 Dataset Dimensions**

Bước đầu tiên để nắm bắt quy mô của dữ liệu là kiểm tra chiều không gian (số lượng quan sát và số lượng đặc trưng).

In [3]:
num_rows_occupancy, num_cols_occupancy = df_occupancy.shape
print(f"Number of rows: {num_rows_occupancy}")
print(f"Number of columns: {num_cols_occupancy}")

Number of rows: 10129
Number of columns: 19


Bộ dữ liệu **Room Occupancy Estimation** bao gồm *10.129 quan sát* và *19 biến đặc trưng*. Cụ thể, không gian đặc trưng (feature space) bao gồm 16 biến liên tục (đo lường các yếu tố môi trường như nhiệt độ, ánh sáng, âm thanh, nồng độ CO2 và tín hiệu chuyển động), 2 biến thời gian (`Date`, `Time`) và 1 biến mục tiêu (`Room_Occupancy_Count`).

Khác với bài toán hồi quy ở phần 1, biến mục tiêu ở đây là một biến rời rạc, dao động từ 0 đến 3 (tương ứng với số lượng người hiện diện trong phòng). 

### **3.2 Observational Unit**

Đơn vị quan sát cốt lõi của tập dữ liệu này được xác định như sau:

> **Một bản ghi tương ứng với một mẫu quét (scan) trạng thái môi trường kéo dài 30 giây tại một căn phòng thí nghiệm tiêu chuẩn (kích thước 6m x 4.6m).**

**Đặc điểm cấu trúc dữ liệu:**
Bộ dữ liệu này mang cấu trúc *Chuỗi thời gian tần suất rất cao (High-frequency Time-series)*. Các tín hiệu được thu thập từ một hệ thống mạng lưới cảm biến IoT đa phương thức (multimodal sensor network bao gồm 7 cụm cảm biến) hoạt động liên tục trong suốt 4 ngày.

### **3.3 Initial Data Glimpse**

Để có cái nhìn ban đầu về cấu trúc và đặc điểm của dataset, chúng em hiển thị một số dòng đầu và cuối để kiểm tra định dạng dữ liệu và mức độ nhất quán:

In [4]:
# Xem 5 dòng đầu
df_occupancy.head()

,Date,Time,S1_Temp,S2_Temp,S3_Temp,S4_Temp,S1_Light,S2_Light,S3_Light,S4_Light,S1_Sound,S2_Sound,S3_Sound,S4_Sound,S5_CO2,S5_CO2_Slope,S6_PIR,S7_PIR,Room_Occupancy_Count
0,22-12-2017,10:49:41,24.94,24.75,24.56,25.38,121,34,53,40,0.08,0.19,0.06,0.06,390,0.769231,0,0,1
1,22-12-2017,10:50:12,24.94,24.75,24.56,25.44,121,33,53,40,0.93,0.05,0.06,0.06,390,0.646154,0,0,1
2,22-12-2017,10:50:42,25.00,24.75,24.50,25.44,121,34,53,40,0.43,0.11,0.08,0.06,390,0.519231,0,0,1
3,22-12-2017,10:51:13,25.00,24.75,24.56,25.44,121,34,53,40,0.41,0.10,0.10,0.09,390,0.388462,0,0,1
4,22-12-2017,10:51:44,25.00,24.75,24.56,25.44,121,34,54,40,0.18,0.06,0.06,0.06,390,0.253846,0,0,1


In [5]:
# Xem 5 dòng cuối
df_occupancy.tail()

,Date,Time,S1_Temp,S2_Temp,S3_Temp,S4_Temp,S1_Light,S2_Light,S3_Light,S4_Light,S1_Sound,S2_Sound,S3_Sound,S4_Sound,S5_CO2,S5_CO2_Slope,S6_PIR,S7_PIR,Room_Occupancy_Count
10124,11-01-2018,08:58:07,25.06,25.13,24.69,25.31,6,7,33,22,0.09,0.04,0.06,0.08,345,0.0,0,0,0
10125,11-01-2018,08:58:37,25.06,25.06,24.69,25.25,6,7,34,22,0.07,0.05,0.05,0.08,345,0.0,0,0,0
10126,11-01-2018,08:59:08,25.13,25.06,24.69,25.25,6,7,34,22,0.11,0.05,0.06,0.08,345,0.0,0,0,0
10127,11-01-2018,08:59:39,25.13,25.06,24.69,25.25,6,7,34,22,0.08,0.08,0.10,0.08,345,0.0,0,0,0
10128,11-01-2018,09:00:09,25.13,25.06,24.69,25.25,6,7,34,22,0.08,0.05,0.06,0.08,345,0.0,0,0,0


**Summary of Column Types**

In [6]:
df_occupancy.dtypes.value_counts()

float64    9
int64      8
object     2
Name: count, dtype: int64

Kết quả phân tích cho thấy tập dữ liệu `df_occupancy` có tổng cộng 19 thuộc tính. Trong đó, dữ liệu số chiếm tỷ trọng tuyệt đối với 17 biến (`float64` và `int64`), đại diện cho tín hiệu liên tục từ 7 cụm cảm biến IoT vật lý.

Hai biến có kiểu `object` (chuỗi ký tự) là `Date` và `Time`. Việc chia tách thời gian thành hai cột độc lập ở định dạng chuỗi gây khó khăn cho các phép phân tích chuỗi thời gian. Do đó, bước tiền xử lý bắt buộc ở đây là phải nối hai cột này lại và ép kiểu (type-casting) về một cột `Datetime` chuẩn của Pandas.

### **3.4 Temporal Coverage & Spatial Distribution**

Để đánh giá chính xác độ phủ thời gian, ta tiến hành gộp cột `Date` và `Time`, sau đó ép kiểu với định dạng chuẩn:

In [7]:
df_occupancy['Datetime'] = pd.to_datetime(df_occupancy['Date'] + ' ' + df_occupancy['Time'], format='%d-%m-%Y %H:%M:%S')
print(df_occupancy['Datetime'].min(), "đến", df_occupancy['Datetime'].max())

2017-12-22 10:49:41 đến 2018-01-11 09:00:09


**Temporal Coverage:** Dữ liệu trải dài liên tục từ ngày `22/12/2017` đến sáng ngày `11/01/2018` (gần 20 ngày). Khoảng thời gian này vô cùng giá trị vì nó bao phủ nhiều tuần lễ liên tiếp, cho phép thuật toán học được các chu kỳ sử dụng phòng rõ rệt: sự khác biệt giữa ngày thường và ngày nghỉ, cũng như khoảng thời gian trống phòng trong kỳ nghỉ lễ Giáng sinh và Năm mới. Kết hợp với tần suất lấy mẫu dày đặc (30 giây/lần), bộ dữ liệu vừa có khả năng bắt được xu hướng vĩ mô (theo tuần), vừa nhận diện được các sự kiện vi mô (người ra vào tức thời).

**Spatial Distribution:** Toàn bộ thực nghiệm được thiết lập trong một không gian kín duy nhất (phòng thí nghiệm kích thước 6m x 4.6m). Việc cô lập không gian này giúp loại bỏ hoàn toàn các yếu tố ngoại cảnh nhiễu loạn, đảm bảo rằng mọi biến thiên về ánh sáng, âm thanh hay nồng độ khí đều do các hoạt động bên trong căn phòng (sự hiện diện của con người) tạo ra.

## **4. Data Semantics**

Phần này phân tích ngữ nghĩa của dữ liệu nhằm vạch ra chiến lược trích xuất đặc trưng phù hợp cho mô hình Phân lớp.

### **4.1 The meaning of each row**

Mỗi dòng trong bộ dữ liệu **Room Occupancy Estimation** đại diện cho một *snapshot trạng thái vật lý của căn phòng trong một cửa sổ thời gian 30 giây*.

Trong 30 giây này, hệ thống ghi nhận:

1. Mức độ ánh sáng, nhiệt độ, âm thanh từ 4 cụm cảm biến quanh phòng.
2. Tín hiệu hồng ngoại (phát hiện thân nhiệt chuyển động) từ 2 cảm biến PIR.
3. Nồng độ CO2 và tốc độ biến thiên CO2 từ 1 cụm cảm biến không khí.
4. Nhãn thực tế (Ground truth): Số lượng người đang có mặt trong phòng tại giây thứ 30 đó.

**Tính chất:** Đây là dữ liệu *Multivariate Time-series* phục vụ cho bài toán *Phân lớp đa lớp (Multiclass Classification)*.

**Lưu ý:** Các biến trong tập dữ liệu này có độ nhạy thời gian rất khác nhau. Ánh sáng, âm thanh và chuyển động (PIR) sẽ phản ứng *ngay lập tức* khi có người bước vào. Ngược lại, nồng độ CO2 và nhiệt độ có *độ trễ (lag)*, chúng sẽ tăng từ từ. Do đó, giống như bài toán hồi quy trước, ta tuyệt đối *không được dùng random shuffle* dữ liệu khi chia tập Train/Test để không phá vỡ tính liên tục của hệ sinh thái cảm biến này.

### **4.2 The meaning of each column**

In [8]:
df_occupancy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10129 entries, 0 to 10128
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   Date                  10129 non-null  object        
 1   Time                  10129 non-null  object        
 2   S1_Temp               10129 non-null  float64       
 3   S2_Temp               10129 non-null  float64       
 4   S3_Temp               10129 non-null  float64       
 5   S4_Temp               10129 non-null  float64       
 6   S1_Light              10129 non-null  int64         
 7   S2_Light              10129 non-null  int64         
 8   S3_Light              10129 non-null  int64         
 9   S4_Light              10129 non-null  int64         
 10  S1_Sound              10129 non-null  float64       
 11  S2_Sound              10129 non-null  float64       
 12  S3_Sound              10129 non-null  float64       
 13  S4_Sound        

Không gian đặc trưng gồm 19 biến được phân loại thành 6 nhóm ngữ nghĩa chính dựa trên bản chất vật lý của cảm biến:

**1. Biến thời gian (Temporal Features):**
* `Date`, `Time`: Thời điểm ghi nhận. Sau khi gộp thành `Datetime`, ta có thể trích xuất giờ trong ngày (để xác định giờ làm việc/giờ nghỉ), giúp mô hình có thêm ngữ cảnh dự báo.

**2. Biến mục tiêu (Target Variable):**
* `Room_Occupancy_Count`: **(Biến mục tiêu - Target)** Số lượng người hiện diện trong phòng, nhận các giá trị rời rạc [0, 1, 2, 3]. Đây là bài toán phân lớp đa lớp.

**3. Nhóm biến Phản ứng tức thời:**
Đây là những tín hiệu biến thiên ngay lập tức khi trạng thái phòng thay đổi:
* **Ánh sáng:** `S1_Light`, `S2_Light`, `S3_Light`, `S4_Light`. Bật/tắt đèn hoặc bóng người che khuất cảm biến sẽ làm thay đổi chỉ số này lập tức.
* **Âm thanh:** `S1_Sound`, `S2_Sound`, `S3_Sound`, `S4_Sound`. Đo lường tiếng ồn (bước chân, giọng nói).
* **Chuyển động:** `S6_PIR`, `S7_PIR`. Cảm biến hồng ngoại thụ động nhận diện thân nhiệt di chuyển. Giá trị dạng nhị phân hoặc đếm nhịp. *(Dự báo: Các biến này sẽ đóng vai trò cực kỳ quan trọng, tuy nhiên có thể bị "đánh lừa" nếu người trong phòng ngồi hoàn toàn bất động).*

**4. Nhóm biến Phản ứng có độ trễ:**
Đây là những tín hiệu thay đổi dựa trên sự tích lũy theo thời gian:
* **Nhiệt động lực học:** `S1_Temp`, `S2_Temp`, `S3_Temp`, `S4_Temp`. Thân nhiệt con người tỏa ra sẽ làm tăng nhiệt độ phòng, nhưng quá trình này diễn ra rất chậm.
* **Chất lượng không khí:** `S5_CO2` (Nồng độ CO2). Lượng người hô hấp càng nhiều, CO2 càng tăng.

**5. Nhóm biến Động lực học:**
* `S5_CO2_Slope`: Tốc độ thay đổi của nồng độ CO2 (Đạo hàm bậc 1 của `S5_CO2`). Đây là một đặc trưng (feature) được các nhà nghiên cứu tạo sẵn cực kỳ giá trị. Khi độ dốc này dương mạnh, phòng đang có người bước vào; khi độ dốc âm, người đang rời đi. Nó bù đắp hoàn hảo cho điểm yếu "độ trễ" của biến CO2 nguyên bản.

### **4.3 Column Data Types and Compatibility for Analysis**

In [9]:
dtype_summary = df_occupancy.dtypes.value_counts().reset_index()
dtype_summary.columns = ['Data Type', 'Count']
dtype_summary['Example Features'] = dtype_summary['Data Type'].apply(
    lambda x: list(df_occupancy.select_dtypes(include=x).columns[:3])
)

print("Data Type Distribution:")
display(dtype_summary)
print("\nDetailed Feature Types:")
numerical_feats = df_occupancy.select_dtypes(include=['float64', 'int64']).columns
categorical_feats = df_occupancy.select_dtypes(include=['object']).columns

print(f" - Numerical Features ({len(numerical_feats)}): {list(numerical_feats)}")
print(f" - Categorical Features ({len(categorical_feats)}): {list(categorical_feats)}")

Data Type Distribution:


,Data Type,Count,Example Features
0,float64,9,"[S1_Temp, S2_Temp, S3_Temp]"
1,int64,8,"[S1_Light, S2_Light, S3_Light]"
2,object,2,"[Date, Time]"
3,datetime64[ns],1,[Datetime]



Detailed Feature Types:
 - Numerical Features (17): ['S1_Temp', 'S2_Temp', 'S3_Temp', 'S4_Temp', 'S1_Light', 'S2_Light', 'S3_Light', 'S4_Light', 'S1_Sound', 'S2_Sound', 'S3_Sound', 'S4_Sound', 'S5_CO2', 'S5_CO2_Slope', 'S6_PIR', 'S7_PIR', 'Room_Occupancy_Count']
 - Categorical Features (2): ['Date', 'Time']


Kết quả phân tích cho thấy cấu trúc của tập dữ liệu `df_occupancy` có sự phân mảnh về kiểu dữ liệu, đòi hỏi một chiến lược tiền xử lý chuyên biệt cho từng nhóm biến trước khi đưa vào mô hình Phân lớp.

**Nhóm dữ liệu định lượng (Numerical - float64 & int64):** Chiếm đa số với 17 biến, biểu diễn các tín hiệu cảm biến liên tục và biến mục tiêu.
* *Định hướng xử lý 1 (Với biến đặc trưng):* Mặc dù đều là dạng số, thang đo (scale) của các biến này có sự chênh lệch khổng lồ. Ví dụ: `S5_CO2` có giá trị lên tới hàng ngàn (ppm), trong khi `S1_Sound` lại là số thập phân rất nhỏ, và biến `PIR` chỉ mang giá trị nhị phân (0/1). Do đó, *nên thực hiện chuẩn hóa dữ liệu (Standardization hoặc Min-Max Scaling)* để các mô hình phân lớp như Support Vector Machine (SVM) hay Logistic Regression không bị thiên lệch trọng số về phía biến CO2.
* *Định hướng xử lý 2 (Với biến mục tiêu):* Cột `Room_Occupancy_Count` mang kiểu `int64`, tuy nhiên bản chất toán học của nó không phải là biến liên tục mà là *biến phân loại thứ bậc (Ordinal Categorical Variable)* đại diện cho các class [0, 1, 2, 3]. Thuật toán học máy cần được cấu hình rõ đây là bài toán Classification (nhận diện lớp) chứ không phải Regression (dự báo số lượng).

**Nhóm dữ liệu chuỗi nguyên bản (Categorical - object):** Gồm 2 cột `Date` và `Time`.
* *Định hướng xử lý:* Vì chúng ta đã khởi tạo thành công cột `Datetime` tổng hợp, hai cột gốc này đã trở nên dư thừa. Giữ lại chúng sẽ gây nhiễu cho các thuật toán. Cần tiến hành loại bỏ 2 cột này khỏi Design Matrix trong bước tiền xử lý.

**Nhóm dữ liệu thời gian (Datetime - datetime64[ns]):** Biến `Datetime`.
* *Định hướng xử lý:* Hoạt động chiếm dụng phòng phụ thuộc mật thiết vào lịch trình sinh hoạt. Tương tự như bài toán năng lượng, ta cần áp dụng kỹ thuật Feature Engineering để trích xuất các tri thức ẩn: `Hour_of_day` (để nhận diện giờ làm việc), `Day_of_week`, và đặc biệt là cờ `is_weekend` (nhận giá trị 1 nếu là T7/CN, 0 nếu là ngày thường) để giúp mô hình dễ dàng đoán được phòng trống (Class 0) vào cuối tuần.

## **5. Descriptive Statistics**

 Thống kê mô tả: mean, std, min, max, tứ phân vị.

## **6. Target Variable Distribution And Class Imbalance Analysis**

Phân bố của biến mục tiêu (histogram, boxplot).

*(Bao gồm: Vẽ Histogram để xem hình dáng phân bố, Boxplot để xem phổ giá trị của biến mục tiêu.*

Mô tả: Sử dụng Bar Chart/Countplot để đếm số lượng quan sát của từng lớp (0, 1, 2, 3 người).

Trong thực tế, phòng trống (Class 0) sẽ chiếm đại đa số thời gian (có thể lên tới 70-80% do qua đêm và cuối tuần). Bạn cần tính toán rõ tỷ lệ phần trăm (%), chỉ ra lớp thiểu số (Minority Class - ví dụ lớp 3 người) và khẳng định đây là bài toán Imbalanced Multiclass Classification. Quyết định này sẽ dẫn đường cho việc chọn thang đo đánh giá mô hình sau này (dùng F1-Macro thay vì Accuracy).

## **7. Correlation & Bivariate Analysis**

*(Bao gồm: Vẽ Heatmap cho Ma trận tương quan (Correlation Matrix) để tìm ra các biến môi trường/thời tiết có ảnh hưởng mạnh nhất đến biến mục tiêu. Vẽ Scatter plot cho các cặp biến quan trọng nhất để xem mối quan hệ là tuyến tính hay phi tuyến).*

### **7.1. Preliminary Patterns: Correlation Matrix**

Trước tiên, nhóm tiến hành tính toán ma trận tương quan cho các biến số để xem xét mức độ liên kết tuyến tính giữa chúng.

In [10]:
# correlation_matrix_heatmap(df_weather)

Ma trận tương quan Pearson này giúp chúng ta định lượng mối quan hệ tuyến tính giữa các biến số. Từ Heatmap, nhóm chia các biến thành các "cụm thông tin" (information clusters) có tác động qua lại mạnh mẽ.

#### **7.1.1. Cụm Đa cộng tuyến cực mạnh (High Multicollinearity)**
Đây là các nhóm biến chứa thông tin gần như trùng lặp, cần lưu ý khi xây dựng các mô hình nhạy cảm với đa cộng tuyến như Logistic Regression.
* Nhóm Nhiệt độ (`MinTemp`, `MaxTemp`, `Temp9am`, `Temp3pm`):
  * Các biến này có hệ số tương quan thuận từ 0.70 đến 0.98.
  * Đáng chú ý nằm ở 2 biến `MaxTemp` và `Temp3pm` có tương quan gần như tuyệt đối ($\approx 0.98$). Điều này hợp lý vì nhiệt độ cao nhất trong ngày thường rơi vào khoảng xế chiều.
  * Nhóm hướng đến việc xử lí bằng cách trong các bước Feature Selection, chỉ cần giữ lại `MaxTemp` hoặc tính toán biến `TempRange = MaxTemp - MinTemp` để nắm bắt sự biến động nhiệt độ thay vì dùng cả 4 biến.
* Nhóm Áp suất (`Pressure9am`, `Pressure3pm`):
  * Có hệ số tương quan thuận cực mạnh (0.96). Điều này cho thấy áp suất khí quyển thường thay đổi rất chậm trong ngày.
  * Hướng xử lí: Có thể sử dụng giá trị trung bình hoặc chỉ giữ lại một biến vì chúng cung cấp thông tin dự báo tương đương nhau.

#### **7.1.2. Mối quan hệ nghịch biến giữa Sunshine, Cloud và Humidity**
Đây là nhóm biến chứa thông tin quan trọng để dự báo khả năng có mưa.
* **Sunshine với Cloud3pm/Cloud9am**:
  * Hệ số tương quan nghịch mạnh (khoảng -0.67 đến -0.71).
  * Lý do là vì lượng mây che phủ càng lớn thì số giờ nắng càng thấp. Mối quan hệ này rất chặt chẽ và nhất quán, cho thấy độ tin cậy của dữ liệu thu thập.
* **Sunshine với Humidity3pm**:
  * Hệ số tương quan nghịch khoảng -0.62.
  * Nhận xét: Khi độ ẩm buổi chiều cao, thường đi kèm với việc hình thành mây, dẫn đến giảm số giờ nắng. Đây là dấu hiệu tiền đề của các cơn mưa vào ngày hôm sau.

#### **7.1.3. Sự khác biệt giữa các mốc thời gian (9am và 3pm)**
* **Humidity**: `Humidity3pm` thường có tương quan với khả năng mưa cao hơn so với `Humidity9am`.
* **Wind Speed**: `WindGustSpeed` (tốc độ gió giật cao nhất) có tương quan thuận với cả `WindSpeed9am` và `WindSpeed3pm`, nhưng thường mạnh hơn ở mốc 3pm. Điều này gợi ý rằng các biến số đo lường vào buổi chiều mang nhiều tín hiệu dự báo cho ngày mai hơn là các biến buổi sáng.

#### **7.1.4. Những mối tương quan yếu gây bất ngờ**
* **Rainfall với các biến khác**: Lượng mưa của ngày hôm nay (`Rainfall`) có tương quan khá thấp ($< 0.3$) với các biến số khác như Áp suất hay Nhiệt độ. Điều này có thể lí giải bởi trường hợp mưa là một hiện tượng phi tuyến tính phức tạp. Việc hôm nay mưa bao nhiêu không phụ thuộc tuyến tính đơn giản vào việc hôm nay nóng bao nhiêu. Điều này cho thấy việc sử dụng các mô hình học máy phi tuyến (như Random Forest hoặc XGBoost) sẽ hiệu quả hơn các mô hình tuyến tính đơn giản.
* **WindGustSpeed với Pressure**: Có hệ số tương quan nghịch nhẹ, cho thấy rằng khi áp suất giảm đột ngột (rãnh thấp), tốc độ gió thường có xu hướng tăng lên.

## **8. Outlier Detection** 

*(Bao gồm: Sử dụng phương pháp IQR hoặc Z-score trên các biến quan trọng. Giải thích nguyên nhân xuất hiện ngoại lai (do lỗi cảm biến hay do thực tế sử dụng) và đề xuất hướng xử lý: giữ nguyên, cắt xén (clipping), hoặc loại bỏ).*

## **9. Data Preprocessing & Feature Engineering**

### **9.1 Missing Data**

Phân tích dữ liệu thiếu (missing data) là bước then chốt để đánh giá độ tin cậy của bộ dữ liệu. Mục tiêu của phần này không chỉ là định lượng tỷ lệ thiếu, mà quan trọng hơn là xác định Cơ chế gây thiếu (Missingness Mechanism). Việc xác định dữ liệu thiếu ngẫu nhiên (MCAR) hay có hệ thống (MNAR) sẽ quyết định chiến lược xử lý (Imputation Strategy), đảm bảo không làm sai lệch tính chất vật lý của dữ liệu khí tượng.

#### **Missing Summary Table**

Chúng ta bắt đầu bằng cái nhìn tổng quan định lượng về tỷ lệ thiếu của từng biến số.

#### **Missing Pattern Visualization**

Để kiểm chứng giả thuyết về "Structural Missingness", ta sử dụng các biểu đồ trực quan để xem xét sự phân bố không gian và sự tương quan của các giá trị thiếu.

### **9.2 Temporal Feature Engineering**

* **Nội dung:** Từ cột `Datetime`, tạo ra các biến phái sinh mang ý nghĩa sinh hoạt:
* `Hour_of_day`: Giờ trong ngày (0-23).
* `Day_of_week`: Ngày trong tuần (Thứ 2 - CN).
* `Is_Weekend`: Cờ nhị phân (1 = Cuối tuần, 0 = Ngày thường).


Giải thích rằng các mô hình Machine Learning không hiểu chuỗi ngày tháng dài, nhưng nó sẽ hiểu quy luật "nếu là cuối tuần (`Is_Weekend` = 1) và ngoài giờ hành chính, xác suất phòng trống (Class 0) gần như tuyệt đối". Đây là bước biến tri thức chuyên gia (Domain Knowledge) thành dữ liệu.

### **9.3 Data Splitting Strategy**

* **Nội dung:** Chia tập Train/Test (ví dụ: 80% Train, 20% Test).
* **Điểm nhấn chuyên sâu (Cực kỳ quan trọng):** Nhấn mạnh việc thiết lập tham số `shuffle=False` (không xáo trộn ngẫu nhiên). Nếu bạn xáo trộn, mô hình sẽ dùng dữ liệu của ngày hôm sau để dự báo ngày hôm trước $\rightarrow$ phá vỡ hoàn toàn ý nghĩa thời gian của biến `S5_CO2_Slope` (tốc độ biến thiên CO2) và gây Data Leakage.

### **9.4 Feature Scaling**

* **Nội dung:** Áp dụng `StandardScaler` (hoặc `MinMaxScaler`) cho 17 biến định lượng.
* **Điểm nhấn chuyên sâu:** Chỉ định rõ hàm `.fit_transform()` CHỈ ĐƯỢC CHẠY trên tập Train. Đối với tập Test, ta chỉ dùng `.transform()` (áp dụng bộ tỷ lệ đã học từ Train sang) để mô phỏng điều kiện thực tế khi mô hình gặp dữ liệu hoàn toàn mới.

### **9.5 Class Imbalance Mitigation Strategy**

* **Nội dung:** Giải quyết vấn đề đã nêu ở mục 6. Ở đây, bạn có 2 hướng xử lý chuyên sâu:
* *Cách 1 (Xử lý ở cấp độ Dữ liệu):* Dùng kỹ thuật Resampling như **SMOTE** (Tạo mẫu tổng hợp) trên tập **Train** (Tuyệt đối không SMOTE trên tập Test). *Lưu ý: SMOTE với chuỗi thời gian khá phức tạp vì có thể tạo ra các điểm dữ liệu không hợp logic vật lý.*
* *Cách 2 (Xử lý ở cấp độ Thuật toán - Khuyên dùng):* Không thay đổi dữ liệu gốc, mà cấu hình tham số `class_weight='balanced'` khi khai báo các mô hình (Random Forest, SVM, Logistic Regression). Kỹ thuật này sẽ phạt nặng (penalize) mô hình nếu nó đoán sai các lớp thiểu số (1, 2, 3 người). Giải thích lý do chọn Cách 2 sẽ cho thấy sự uyên bác của bạn.

---

# **DATA MODELING AND MODEL EVALUATION**